# Validación del conjunto de datos limpio

Se comprueba que:
- No existan registros duplicados exactos.
- No existan espacios al inicio o final de textos.
- Los teléfonos tengan un formato consistente.
- Los departamentos (y, cuando el catálogo esté disponible, los municipios) pertenezcan al catálogo oficial.
- Las variables tengan el tipo de dato esperado.
- No existan categorías duplicadas por diferencias de escritura.
- No existan los valores inválidos detectados durante el diagnóstico inicial.


In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from difflib import SequenceMatcher
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

VALIDACIONES = []

def validar(prueba, criterio, resultado_ok, detalle, n_afectados=0):
    VALIDACIONES.append({
        "Prueba": prueba,
        "Criterio": criterio,
        "Resultado": "PASA" if resultado_ok else "FALLA",
        "Registros_afectados": n_afectados,
        "Detalle": detalle,
    })
    estado = "PASA" if resultado_ok else "FALLA"
    print(f"[{estado}] {prueba} -> {detalle}")


## Carga del conjunto limpio

Se carga el resultado de `02_limpieza.ipynb`. También se cargan el conjunto crudo y la tabla de
transformaciones para poder contrastar cifras "antes vs. después" cuando aplique.


In [2]:
df = pd.read_csv("../data/processed/centros_educativos_gt_limpio.csv", dtype=str, encoding="utf-8-sig")
print("Filas:", len(df), " Columnas:", len(df.columns))
print("Columnas:", list(df.columns))
df.head(3)


Filas: 92228  Columnas: 23
Columnas: ['CODIGO', 'DISTRITO', 'DEPARTAMENTO', 'MUNICIPIO', 'ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR', 'NIVEL', 'SECTOR', 'AREA', 'STATUS', 'MODALIDAD', 'JORNADA', 'PLAN', 'DEPARTAMENTAL', 'ESTABLECIMIENTO_SIGLA', 'ZONA', 'DISTRITO_FORMATO_ATIPICO', 'TELEFONO_ADICIONAL', 'GRUPO_DUPLICADO_ID', 'TIPO_DUPLICADO']


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,ESTABLECIMIENTO_SIGLA,ZONA,DISTRITO_FORMATO_ATIPICO,TELEFONO_ADICIONAL,GRUPO_DUPLICADO_ID,TIPO_DUPLICADO
0,16-01-0026-45,16-01-0923,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO IMPERIAL,5A. CALLE 1-98 ZONA 3,57101061,EFRAIN CAAL CUC,MYNOR GUSTAVO IPIÑA ESPAÑA,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,DOBLE,FIN DE SEMANA,ALTA VERAPAZ,NaN,NaN,False,NaN,1,oferta_multiple
1,16-01-0135-45,16-01-0925,ALTA VERAPAZ,COBAN,INEB ADSCRITO A INSTITUTO 'EMILIO ROSALES PONCE',3A AVE 6-23 ZONA 11,54457786,NORA LILIANA FIGUEROA HERNÁNDEZ,VICTOR HUGO DOMÍNGUEZ REYES,BASICO,OFICIAL,URBANA,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,NaN,NaN,False,NaN,NaN,NaN
2,16-01-0136-45,16-01-0923,ALTA VERAPAZ,COBAN,INEB,6A AVE 1-15 ZONA 4,55541910,EFRAIN CAAL CUC,MAYRA PAULINA LAZARO CABRERA,BASICO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,NaN,NaN,False,NaN,NaN,NaN


In [3]:
df_crudo = pd.read_csv("../data/processed/centros_educativos_gt.csv", dtype=str, encoding="utf-8", header=1)
tabla_transformaciones = pd.read_csv("../transformations/registro_transformaciones.csv")

print("Filas crudo:", len(df_crudo), " Filas limpio:", len(df))
tabla_transformaciones


Filas crudo: 101224  Filas limpio: 92228


,Variable,Problema detectado,Transformación,Registros afectados,Justificación
0,Todas (fila completa),Filas de encabezado de cada CSV departamental ...,Se eliminaron 181 filas donde CODIGO == 'CODIGO'.,181,No son establecimientos reales; son un artefac...
1,DIRECCION,"Placeholders de dato faltante (espacios, símbo...",Se convirtieron 28 valores placeholder a NA ex...,28,Unifica el conteo real de valores faltantes; v...
2,SUPERVISOR,"Placeholders de dato faltante (espacios, símbo...",Se convirtieron 22 valores placeholder a NA ex...,22,Unifica el conteo real de valores faltantes; v...
3,DIRECTOR,"Placeholders de dato faltante (espacios, símbo...",Se convirtieron 2686 valores placeholder a NA ...,2686,Unifica el conteo real de valores faltantes; v...
4,Todas (texto),"Espacios al inicio/fin, espacios múltiples int...",Trim + colapso de espacios múltiples + normali...,0,Estandariza la representación de texto sin alt...
5,ESTABLECIMIENTO,Comillas dobles embebidas en el nombre del est...,Se eliminó el carácter de comillas dobles en 6...,6149,Las comillas no aportan significado y dificult...
6,ESTABLECIMIENTO_SIGLA (derivada),El nombre del establecimiento a veces incluye ...,Se creó la variable derivada ESTABLECIMIENTO_S...,3888,Facilita búsquedas/agrupaciones por sigla sin ...
7,DEPARTAMENTO / MUNICIPIO,'CIUDAD CAPITAL' no es un departamento oficial...,Se remapeó DEPARTAMENTO 'CIUDAD CAPITAL' -> 'G...,7782,Alinea DEPARTAMENTO con el catálogo oficial de...
8,DEPARTAMENTAL,No corresponde 1:1 con DEPARTAMENTO (subdivide...,Se conserva sin fusionar con DEPARTAMENTO; sol...,40960,NOTA: requeriría confirmar con el MINEDUC si e...
9,DISTRITO,Conviven al menos dos formatos de codificación...,No se fuerza un único formato (se documentan a...,79,NOTA: requeriría confirmar con el MINEDUC el s...


## Duplicados exactos


In [4]:
n_dup_exactos = int(df.duplicated().sum())
n_codigo_dup = int(df["CODIGO"].duplicated().sum())

validar(
    "Duplicados exactos (fila completa)",
    "df.duplicated().sum() == 0",
    n_dup_exactos == 0,
    f"{n_dup_exactos} filas completamente duplicadas encontradas.",
    n_dup_exactos,
)

validar(
    "CODIGO único",
    "df['CODIGO'].duplicated().sum() == 0",
    n_codigo_dup == 0,
    f"{n_codigo_dup} valores de CODIGO repetidos encontrados.",
    n_codigo_dup,
)


[PASA] Duplicados exactos (fila completa) -> 0 filas completamente duplicadas encontradas.
[PASA] CODIGO único -> 0 valores de CODIGO repetidos encontrados.


In [5]:
duplicados = df[df.duplicated(keep=False)]

duplicados.sort_values("CODIGO").head(20)

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,ESTABLECIMIENTO_SIGLA,ZONA,DISTRITO_FORMATO_ATIPICO,TELEFONO_ADICIONAL,GRUPO_DUPLICADO_ID,TIPO_DUPLICADO


## Espacios al inicio o final de textos


In [6]:
columnas_texto = df.select_dtypes(include="object").columns

espacios_por_columna = {}
for col in columnas_texto:
    n = df[col].apply(lambda x: isinstance(x, str) and x != x.strip()).sum()
    n += df[col].astype(str).str.contains(r"\s{2,}", regex=True, na=False).sum()
    if n > 0:
        espacios_por_columna[col] = int(n)

n_total_espacios = sum(espacios_por_columna.values())

validar(
    "Espacios al inicio/final o múltiples",
    "Ninguna columna de texto con valores != valor.strip() ni espacios dobles internos",
    n_total_espacios == 0,
    f"{n_total_espacios} celdas con espacios sobrantes en columnas: {list(espacios_por_columna.keys())}" if n_total_espacios else "Sin celdas con espacios sobrantes.",
    n_total_espacios,
)

if espacios_por_columna:
    display(pd.Series(espacios_por_columna, name="celdas_afectadas"))


[PASA] Espacios al inicio/final o múltiples -> Sin celdas con espacios sobrantes.


## Formato de teléfono


In [7]:
patron_telefono = re.compile(r"^\d{7,8}$")

telefono_no_na = df["TELEFONO"].dropna()
telefono_invalido = telefono_no_na[~telefono_no_na.str.match(patron_telefono)]

validar(
    "Formato de TELEFONO",
    r"TELEFONO coincide con ^\d{7,8}$ o es NA",
    len(telefono_invalido) == 0,
    f"{len(telefono_invalido)} valores de TELEFONO con formato inválido." if len(telefono_invalido) else "Todos los valores no nulos tienen 7 u 8 dígitos.",
    len(telefono_invalido),
)

adicional_no_na = df["TELEFONO_ADICIONAL"].dropna()
def todos_validos(cadena):
    return all(patron_telefono.match(n) for n in cadena.split(";"))

adicional_invalido = adicional_no_na[~adicional_no_na.apply(todos_validos)]

validar(
    "Formato de TELEFONO_ADICIONAL",
    "Cada número separado por ';' coincide con ^\\d{7,8}$",
    len(adicional_invalido) == 0,
    f"{len(adicional_invalido)} valores de TELEFONO_ADICIONAL con formato inválido." if len(adicional_invalido) else "Todos los valores no nulos tienen 7 u 8 dígitos por número.",
    len(adicional_invalido),
)

if len(telefono_invalido) > 0:
    display(telefono_invalido.head(10))


[PASA] Formato de TELEFONO -> Todos los valores no nulos tienen 7 u 8 dígitos.
[PASA] Formato de TELEFONO_ADICIONAL -> Todos los valores no nulos tienen 7 u 8 dígitos por número.


##  `DEPARTAMENTO` 

In [8]:
CATALOGO_DEPARTAMENTOS = {
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA", "EL PROGRESO",
    "ESCUINTLA", "GUATEMALA", "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA",
    "PETEN", "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ", "SAN MARCOS",
    "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ", "TOTONICAPAN", "ZACAPA",
}

depto_invalido = df[~df["DEPARTAMENTO"].isin(CATALOGO_DEPARTAMENTOS)]

validar(
    "DEPARTAMENTO en catálogo oficial (22 departamentos)",
    "DEPARTAMENTO.isin(CATALOGO_DEPARTAMENTOS) para todas las filas",
    len(depto_invalido) == 0,
    f"{len(depto_invalido)} registros con DEPARTAMENTO fuera de catálogo." if len(depto_invalido) else "Todos los DEPARTAMENTO pertenecen al catálogo oficial.",
    len(depto_invalido),
)

if len(depto_invalido) > 0:
    display(depto_invalido["DEPARTAMENTO"].value_counts())


[PASA] DEPARTAMENTO en catálogo oficial (22 departamentos) -> Todos los DEPARTAMENTO pertenecen al catálogo oficial.


## Tipo de dato esperado por variable


In [9]:
patron_codigo = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
codigo_invalido = df[~df["CODIGO"].str.match(patron_codigo)]

validar(
    "Formato de CODIGO",
    r"CODIGO coincide con ^\d{2}-\d{2}-\d{4}-\d{2}$",
    len(codigo_invalido) == 0,
    f"{len(codigo_invalido)} valores de CODIGO con formato inválido." if len(codigo_invalido) else "Todos los CODIGO cumplen el formato esperado.",
    len(codigo_invalido),
)


COLUMNAS_CATEGORICAS = ["NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]

for col in COLUMNAS_CATEGORICAS:
    n_categorias = df[col].nunique(dropna=True)
    validar(
        f"Dominio acotado de {col}",
        f"{col} tiene un número reducido y estable de categorías (variable categórica)",
        n_categorias <= 15,
        f"{col} tiene {n_categorias} categorías únicas: {sorted(df[col].dropna().unique())}",
        0 if n_categorias <= 15 else n_categorias,
    )


[PASA] Formato de CODIGO -> Todos los CODIGO cumplen el formato esperado.
[PASA] Dominio acotado de NIVEL -> NIVEL tiene 6 categorías únicas: ['BASICO', 'DIVERSIFICADO', 'PARVULOS', 'PREPRIMARIA BILINGUE', 'PRIMARIA', 'PRIMARIA DE ADULTOS']
[PASA] Dominio acotado de SECTOR -> SECTOR tiene 4 categorías únicas: ['COOPERATIVA', 'MUNICIPAL', 'OFICIAL', 'PRIVADO']
[PASA] Dominio acotado de AREA -> AREA tiene 3 categorías únicas: ['RURAL', 'SIN ESPECIFICAR', 'URBANA']
[PASA] Dominio acotado de STATUS -> STATUS tiene 6 categorías únicas: ['ABIERTA', 'CERRADA DEFINITIVAMENTE', 'CERRADA TEMPORALMENTE', 'TEMPORAL CONTRATO 021', 'TEMPORAL NOMBRAMIENTO', 'TEMPORAL TITULOS']
[PASA] Dominio acotado de MODALIDAD -> MODALIDAD tiene 2 categorías únicas: ['BILINGUE', 'MONOLINGUE']
[PASA] Dominio acotado de JORNADA -> JORNADA tiene 6 categorías únicas: ['DOBLE', 'INTERMEDIA', 'MATUTINA', 'NOCTURNA', 'SIN JORNADA', 'VESPERTINA']
[PASA] Dominio acotado de PLAN -> PLAN tiene 13 categorías únicas: ['A DISTAN

In [10]:
niveles_validos = {
    "PARVULOS",
    "PREPRIMARIA BILINGUE",
    "PRIMARIA",
    "PRIMARIA DE ADULTOS",
    "BASICO",
    "DIVERSIFICADO"
}

nivel_invalido = df[
    ~df["NIVEL"].str.upper().isin(niveles_validos)
]

validar(
    "Valores válidos en NIVEL",
    "Todos los registros pertenecen a un nivel educativo reconocido por el MINEDUC.",
    len(nivel_invalido) == 0,
    (
        f"{len(nivel_invalido)} registros presentan un NIVEL no reconocido."
        if len(nivel_invalido)
        else "Todos los registros contienen niveles válidos."
    ),
    len(nivel_invalido),
)


[PASA] Valores válidos en NIVEL -> Todos los registros contienen niveles válidos.


## Categorías duplicadas por diferencias de escritura


In [11]:
def normalizar_para_comparar_categoria(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode("utf-8")
    return x.upper().strip()

def detectar_posibles_duplicados(valores_unicos, umbral=0.88):
    valores = list(valores_unicos)
    normalizados = [normalizar_para_comparar_categoria(v) for v in valores]
    pares_sospechosos = []
    for i in range(len(valores)):
        for j in range(i + 1, len(valores)):
            if normalizados[i] == normalizados[j] and valores[i] != valores[j]:
                pares_sospechosos.append((valores[i], valores[j], 1.0))
            elif SequenceMatcher(None, normalizados[i], normalizados[j]).ratio() >= umbral:
                pares_sospechosos.append((valores[i], valores[j], round(SequenceMatcher(None, normalizados[i], normalizados[j]).ratio(), 3)))
    return pares_sospechosos

COLUMNAS_A_REVISAR = [
    "STATUS",
    "MODALIDAD",
    "JORNADA",
    "SECTOR",
    "AREA"
]

total_sospechosos = 0
for col in COLUMNAS_A_REVISAR:
    sospechosos = detectar_posibles_duplicados(df[col].dropna().unique())
    total_sospechosos += len(sospechosos)
    if sospechosos:
        print(f"\n{col}: posibles categorías duplicadas por escritura:")
        display(pd.DataFrame(sospechosos, columns=["valor_a", "valor_b", "similitud"]))

validar(
    "Categorías duplicadas por diferencias de escritura",
    "Ningún par de valores únicos con similitud >= 0.88 tras normalizar mayúsculas/tildes",
    total_sospechosos == 0,
    f"{total_sospechosos} pares de valores potencialmente duplicados por escritura." if total_sospechosos else "No se detectaron categorías duplicadas por escritura.",
    total_sospechosos,
)


[PASA] Categorías duplicadas por diferencias de escritura -> No se detectaron categorías duplicadas por escritura.


## Valores inválidos detectados en el diagnóstico ya no existen


In [12]:
n_ciudad_capital = int((df["DEPARTAMENTO"] == "CIUDAD CAPITAL").sum())
validar(
    "'CIUDAD CAPITAL' remapeado",
    "No quedan registros con DEPARTAMENTO == 'CIUDAD CAPITAL'",
    n_ciudad_capital == 0,
    f"{n_ciudad_capital} registros aún con 'CIUDAD CAPITAL'." if n_ciudad_capital else "No quedan registros con 'CIUDAD CAPITAL'.",
    n_ciudad_capital,
)

n_comillas = int(df["ESTABLECIMIENTO"].str.contains('"', na=False).sum())
validar(
    "ESTABLECIMIENTO sin comillas embebidas",
    "Ningún valor de ESTABLECIMIENTO contiene el carácter \"",
    n_comillas == 0,
    f"{n_comillas} registros con comillas encontrados." if n_comillas else "Sin comillas embebidas en ESTABLECIMIENTO.",
    n_comillas,
)

telefono_con_letras = df["TELEFONO"].dropna().astype(str).str.contains(r"[A-Za-z]", regex=True)
n_telefono_letras = int(telefono_con_letras.sum())
validar(
    "TELEFONO sin letras",
    "Ningún valor de TELEFONO contiene letras",
    n_telefono_letras == 0,
    f"{n_telefono_letras} valores de TELEFONO con letras." if n_telefono_letras else "Sin letras en TELEFONO.",
    n_telefono_letras,
)

PLACEHOLDERS_TEXTO = {
    "N/A", "NA", "S/N", "S/D", "SIN DATO", "SIN DATOS", "NO TIENE", "NINGUNO",
    "NULO", "NULL", "VACANTE", "NO APLICA", ".",
}
COLUMNAS_TEXTO_LIBRE = ["DISTRITO", "ESTABLECIMIENTO", "DIRECCION", "TELEFONO", "SUPERVISOR", "DIRECTOR"]

n_placeholders_restantes = 0
for col in COLUMNAS_TEXTO_LIBRE:
    n = df[col].dropna().astype(str).str.upper().isin(PLACEHOLDERS_TEXTO).sum()
    n_placeholders_restantes += int(n)

validar(
    "Placeholders de valor faltante convertidos a NA",
    "Ningún valor en columnas de texto libre coincide con placeholders conocidos ('N/A', 'SIN DATO', etc.)",
    n_placeholders_restantes == 0,
    f"{n_placeholders_restantes} placeholders restantes." if n_placeholders_restantes else "Todos los placeholders fueron convertidos a NA.",
    n_placeholders_restantes,
)


fila_distrito = tabla_transformaciones[tabla_transformaciones["Variable"] == "DISTRITO"]
n_atipico_actual = int(df["DISTRITO_FORMATO_ATIPICO"].astype(str).str.upper().eq("TRUE").sum())
n_atipico_esperado = int(fila_distrito["Registros afectados"].iloc[0]) if len(fila_distrito) else None

validar(
    "DISTRITO_FORMATO_ATIPICO consistente con el registro de transformaciones",
    "El conteo de filas marcadas como formato atípico coincide con lo documentado en 02_limpieza.ipynb",
    n_atipico_esperado is not None and n_atipico_actual == n_atipico_esperado,
    f"Actual: {n_atipico_actual}, esperado según registro: {n_atipico_esperado}.",
    abs(n_atipico_actual - (n_atipico_esperado or 0)),
)


[PASA] 'CIUDAD CAPITAL' remapeado -> No quedan registros con 'CIUDAD CAPITAL'.
[PASA] ESTABLECIMIENTO sin comillas embebidas -> Sin comillas embebidas en ESTABLECIMIENTO.
[PASA] TELEFONO sin letras -> Sin letras en TELEFONO.
[PASA] Placeholders de valor faltante convertidos a NA -> Todos los placeholders fueron convertidos a NA.
[PASA] DISTRITO_FORMATO_ATIPICO consistente con el registro de transformaciones -> Actual: 79, esperado según registro: 79.


## Resumen de validación


In [13]:
tabla_validaciones = pd.DataFrame(VALIDACIONES)
tabla_validaciones.to_csv("../transformations/registro_validaciones.csv", index=False)

n_pasa = int((tabla_validaciones["Resultado"] == "PASA").sum())
n_falla = int((tabla_validaciones["Resultado"] == "FALLA").sum())

print(f"Pruebas ejecutadas: {len(tabla_validaciones)}  |  PASA: {n_pasa}  |  FALLA: {n_falla}")
tabla_validaciones


Pruebas ejecutadas: 21  |  PASA: 21  |  FALLA: 0


,Prueba,Criterio,Resultado,Registros_afectados,Detalle
0,Duplicados exactos (fila completa),df.duplicated().sum() == 0,PASA,0,0 filas completamente duplicadas encontradas.
1,CODIGO único,df['CODIGO'].duplicated().sum() == 0,PASA,0,0 valores de CODIGO repetidos encontrados.
2,Espacios al inicio/final o múltiples,Ninguna columna de texto con valores != valor....,PASA,0,Sin celdas con espacios sobrantes.
3,Formato de TELEFONO,"TELEFONO coincide con ^\d{7,8}$ o es NA",PASA,0,Todos los valores no nulos tienen 7 u 8 dígitos.
4,Formato de TELEFONO_ADICIONAL,Cada número separado por ';' coincide con ^\d{...,PASA,0,Todos los valores no nulos tienen 7 u 8 dígito...
5,DEPARTAMENTO en catálogo oficial (22 departame...,DEPARTAMENTO.isin(CATALOGO_DEPARTAMENTOS) para...,PASA,0,Todos los DEPARTAMENTO pertenecen al catálogo ...
6,Formato de CODIGO,CODIGO coincide con ^\d{2}-\d{2}-\d{4}-\d{2}$,PASA,0,Todos los CODIGO cumplen el formato esperado.
7,Dominio acotado de NIVEL,NIVEL tiene un número reducido y estable de ca...,PASA,0,"NIVEL tiene 6 categorías únicas: ['BASICO', 'D..."
8,Dominio acotado de SECTOR,SECTOR tiene un número reducido y estable de c...,PASA,0,SECTOR tiene 4 categorías únicas: ['COOPERATIV...
9,Dominio acotado de AREA,AREA tiene un número reducido y estable de cat...,PASA,0,"AREA tiene 3 categorías únicas: ['RURAL', 'SIN..."


## Conclusiones

El proceso de limpieza y validación permitió obtener un conjunto de datos consistente, documentado y apto para análisis posteriores. A partir del dataset consolidado de establecimientos educativos autorizados por el Ministerio de Educación de Guatemala, correspondiente a todos los niveles educativos, se verificó la calidad de los datos mediante un conjunto de pruebas de validación.

Los principales resultados obtenidos fueron los siguientes:
- **Estructura del conjunto de datos:** se eliminaron las filas fantasma generadas durante la extracción de la información, conservando únicamente registros válidos de establecimientos educativos.
- **Duplicados:** se eliminaron todos los registros completamente duplicados. El conjunto final no presenta filas idénticas ni códigos de establecimiento (`CODIGO`) repetidos como consecuencia de la integración de los diferentes niveles educativos.
- **Calidad del texto:** se eliminaron espacios innecesarios, se normalizó el uso de mayúsculas y caracteres especiales, y se corrigieron las inconsistencias detectadas durante el análisis exploratorio.
- **Variables telefónicas:** todos los valores presentes en `TELEFONO` y `TELEFONO_ADICIONAL` cumplen con el formato esperado de números telefónicos guatemaltecos (7 u 8 dígitos), mientras que los valores faltantes fueron representados de forma consistente.
- **Consistencia geográfica:** el 100% de los registros pertenece a alguno de los 22 departamentos oficiales de Guatemala. Además, el valor `CIUDAD CAPITAL` fue estandarizado correctamente como `GUATEMALA`.
- **Dominios categóricos:** las variables categóricas (`NIVEL`, `SECTOR`, `AREA`, `STATUS`, `MODALIDAD`, `JORNADA` y `PLAN`) presentan únicamente categorías válidas y consistentes con la información publicada por el Ministerio de Educación.
- **Revisión de categorías similares:** se aplicó una detección automática de posibles categorías duplicadas mediante similitud de texto. Los casos identificados correspondían a nombres distintos pero válidos (por ejemplo, municipios diferentes o modalidades distintas), por lo que no fue necesario realizar fusiones adicionales.
- **Valores inválidos identificados durante el diagnóstico:** se verificó que ya no existen registros con valores obsoletos, comillas embebidas en nombres de establecimientos, caracteres alfabéticos en teléfonos ni placeholders de datos faltantes sin transformar.
